In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/SIT788_AISol/FL/

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/SIT788_AISol/FL


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np




In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

# Define the client model
class ClientModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(ClientModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)
        self.dropout = nn.Dropout(0.2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

# Federated Learning Parameters
num_clients = 10  # Number of simulated clients
num_samples_per_client = 100  # Number of training samples per client
input_size = 28 * 28  # Input size (MNIST images are 28x28)
output_size = 10  # Number of classes
num_rounds = 50
local_epochs = 3
learning_rate = 0.01

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and preprocess MNIST data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Split data non-IID among clients
train_data = []
train_labels = []
indices = list(range(len(mnist_train)))

for i in range(num_clients):
    client_indices = indices[i::num_clients]
    X = torch.cat([mnist_train[i][0].view(-1) for i in client_indices])
    y = torch.tensor([mnist_train[i][1] for i in client_indices])
    train_data.append(X.view(-1, 28*28))
    train_labels.append(y)

# Initialize the global model
global_model = ClientModel(input_size, output_size).to(device)

# Perform federated learning rounds
for round in range(num_rounds):
    print(f"Round {round + 1}/{num_rounds}")
    local_weights = []

    global_model.eval()
    for i in range(num_clients):
        # Initialize local model and optimizer
        local_model = ClientModel(input_size, output_size).to(device)
        local_model.load_state_dict(global_model.state_dict())
        optimizer = optim.SGD(local_model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        # Train local model
        local_model.train()
        X = train_data[i].to(device)
        y = train_labels[i].to(device)

        for epoch in range(local_epochs):
            optimizer.zero_grad()
            outputs = local_model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        # Append local model's weights
        local_weights.append({key: param.clone().detach() for key, param in local_model.state_dict().items()})

    # Aggregate weights (FedAvg)
    global_weights = global_model.state_dict()
    for key in global_weights.keys():
        global_weights[key] = torch.mean(torch.stack([local_weights[i][key] for i in range(num_clients)]), dim=0)

    global_model.load_state_dict(global_weights)

# Evaluate the global model
mnist_test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=64, shuffle=False)
global_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X, y in mnist_test_loader:
        X, y = X.view(-1, 28*28).to(device), y.to(device)
        outputs = global_model(X)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print("Global model test accuracy:", accuracy)


Round 1/50
Round 2/50
Round 3/50
Round 4/50
Round 5/50
Round 6/50
Round 7/50
Round 8/50
Round 9/50
Round 10/50
Round 11/50
Round 12/50
Round 13/50
Round 14/50
Round 15/50
Round 16/50
Round 17/50
Round 18/50
Round 19/50
Round 20/50
Round 21/50
Round 22/50
Round 23/50
Round 24/50
Round 25/50
Round 26/50
Round 27/50
Round 28/50
Round 29/50
Round 30/50
Round 31/50
Round 32/50
Round 33/50
Round 34/50
Round 35/50
Round 36/50
Round 37/50
Round 38/50
Round 39/50
Round 40/50
Round 41/50
Round 42/50
Round 43/50
Round 44/50
Round 45/50
Round 46/50
Round 47/50
Round 48/50
Round 49/50
Round 50/50
Global model test accuracy: 0.544
